In [1]:
import sys
print(sys.executable)

c:\Users\George\boston-bike-demand-forecasting\.venv\Scripts\python.exe


In [2]:
# ============================================================
# IMPORT LIBRARIES AND CONFIGURE THE ENVIRONMENT
# ============================================================

import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Locate the project root
project_root = Path().resolve().parents[0]

# Set the working directory
os.chdir(project_root)

# Add the project root to the Python path
sys.path.append(str(project_root))



In [3]:
# ============================================================
# LOAD THE RAW DATASET
# ============================================================

data = pd.read_csv("data/01_raw/blue_bike_boston_raw.csv")

In [4]:
# ============================================================
# STEP 3.1: VERIFY DATASET LOADING
# ============================================================

print("Dataset loaded successfully!")
print(f"Number of rows: {data.shape[0]:,}")
print(f"Number of columns: {data.shape[1]:,}")

Dataset loaded successfully!
Number of rows: 21,079
Number of columns: 15


In [5]:
# ============================================================
# STEP 3.2: INSPECT DATASET STRUCTURE
# ============================================================

# Display the first five observations
display(data.head())

# Inspect column names, data types, and non-null counts
data.info()

,Unnamed: 0,casual_riders_count,member_riders_count,casual_rider_duration,member_rider_duration,count,travel_time,day_of_week,IsWeekend,IsHoliday,Temp(c),rel_humidity,wspd,pres,weather_condition
0,2022-01-01 00:00:00,28.00,97.00,554672.00,96509.00,125.00,651181.00,SATURDAY,1,1,6.70,89.00,7.60,1014.90,Cloudy
1,2022-01-01 01:00:00,31.00,61.00,46276.00,58472.00,92.00,104748.00,SATURDAY,1,1,6.70,89.00,7.60,1014.70,Cloudy
2,2022-01-01 02:00:00,32.00,41.00,377839.00,45191.00,73.00,423030.00,SATURDAY,1,1,6.70,89.00,7.60,1014.70,Cloudy
3,2022-01-01 03:00:00,54.00,41.00,2537734.00,37167.00,95.00,2574901.00,SATURDAY,1,1,6.70,96.00,5.40,1014.20,Cloudy
4,2022-01-01 04:00:00,49.00,42.00,911448.00,43178.00,91.00,954626.00,SATURDAY,1,1,7.20,93.00,5.40,1014.40,Cloudy


<class 'pandas.DataFrame'>
RangeIndex: 21079 entries, 0 to 21078
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             21079 non-null  str    
 1   casual_riders_count    21079 non-null  float64
 2   member_riders_count    21079 non-null  float64
 3   casual_rider_duration  21079 non-null  float64
 4   member_rider_duration  21079 non-null  float64
 5   count                  21079 non-null  float64
 6   travel_time            21079 non-null  float64
 7   day_of_week            21079 non-null  str    
 8   IsWeekend              21079 non-null  int64  
 9   IsHoliday              21079 non-null  int64  
 10  Temp(c)                21079 non-null  float64
 11  rel_humidity           21079 non-null  float64
 12  wspd                   21079 non-null  float64
 13  pres                   21079 non-null  float64
 14  weather_condition      21079 non-null  str    
dtypes: float64(10

In [6]:
# ============================================================
# 3. COLUMN RENAMING AND REORGANISATION
# ============================================================

# Create an explicit mapping from original to standardised names
column_mapping = {
    "Unnamed: 0": "datetime",
    "casual_riders_count": "casual_riders",
    "member_riders_count": "member_riders",
    "casual_rider_duration": "casual_duration",
    "member_rider_duration": "member_duration",
    "count": "total_riders",
    "travel_time": "total_duration",
    "day_of_week": "weekday",
    "IsWeekend": "weekend",
    "IsHoliday": "holiday",
    "Temp(c)": "temperature",
    "rel_humidity": "humidity",
    "wspd": "windspeed",
    "pres": "pressure",
    "weather_condition": "weather_condition",
}

# Validate that all original columns have been accounted for
unmapped_columns = set(data.columns) - set(column_mapping.keys())

if unmapped_columns:
    raise ValueError(
        f"Unmapped columns detected: {sorted(unmapped_columns)}"
    )

# Validate that the mapping does not create duplicate names
new_column_names = list(column_mapping.values())

if len(new_column_names) != len(set(new_column_names)):
    raise ValueError("The mapping produces duplicate column names.")

# Rename the columns
df = data.rename(columns=column_mapping)

# Arrange columns in the desired order
column_order = [
    "datetime",
    "weekday",
    "weekend",
    "holiday",
    "casual_duration",
    "member_duration",
    "total_duration",
    "temperature",
    "humidity",
    "windspeed",
    "pressure",
    "weather_condition",
    "casual_riders",
    "member_riders",
    "total_riders",
    
]

# Verify that all requested columns exist
missing_columns = set(column_order) - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Expected columns are missing: {sorted(missing_columns)}"
    )

# Reorder the dataset
df = df[column_order]

# Display the results
print("Column renaming and reorganisation completed.")
print(f"Dataset shape: {df.shape}")

display(pd.DataFrame({
    "position": range(1, len(df.columns) + 1),
    "column_name": df.columns,
    "data_type": df.dtypes.astype(str).values,
}))

Column renaming and reorganisation completed.
Dataset shape: (21079, 15)


,position,column_name,data_type
0,1,datetime,str
1,2,weekday,str
2,3,weekend,int64
3,4,holiday,int64
4,5,casual_duration,float64
5,6,member_duration,float64
6,7,total_duration,float64
7,8,temperature,float64
8,9,humidity,float64
9,10,windspeed,float64


In [7]:
# ============================================================
# 4. VALIDATE COLUMN TRANSFORMATION
# ============================================================

assert df.shape == data.shape, (
    "Dataset dimensions changed unexpectedly."
)

assert df.columns.is_unique, (
    "Duplicate column names detected."
)

assert df.columns.tolist() == column_order, (
    "Column order does not match the specification."
)

print("All column transformation checks passed.")
print(f"Rows preserved: {len(df):,}")
print(f"Columns preserved: {len(df.columns)}")

All column transformation checks passed.
Rows preserved: 21,079
Columns preserved: 15


In [8]:
df.head(5)

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28.00,97.00,125.00
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31.00,61.00,92.00
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32.00,41.00,73.00
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54.00,41.00,95.00
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49.00,42.00,91.00


In [9]:
# ============================================================
# 4.1 TIMESTAMP INITIAL INSPECTION
# ============================================================

print("Timestamp column:", "datetime")
print("Current data type:", df["datetime"].dtype)

print("\nFirst 10 timestamps:")
display(df["datetime"].head(10))

print("\nLast 10 timestamps:")
display(df["datetime"].tail(10))

print("\nMissing timestamps:")
print(df["datetime"].isna().sum())

print("\nNumber of unique timestamps:")
print(df["datetime"].nunique())

Timestamp column: datetime
Current data type: str

First 10 timestamps:


0    2022-01-01 00:00:00
1    2022-01-01 01:00:00
2    2022-01-01 02:00:00
3    2022-01-01 03:00:00
4    2022-01-01 04:00:00
5    2022-01-01 05:00:00
6    2022-01-01 06:00:00
7    2022-01-01 07:00:00
8    2022-01-01 08:00:00
9    2022-01-01 09:00:00
Name: datetime, dtype: str


Last 10 timestamps:


21069    2024-05-30 15:00:00
21070    2024-05-30 16:00:00
21071    2024-05-30 17:00:00
21072    2024-05-30 18:00:00
21073    2024-05-30 19:00:00
21074    2024-05-30 20:00:00
21075    2024-05-30 21:00:00
21076    2024-05-30 22:00:00
21077    2024-05-30 23:00:00
21078    2024-05-31 00:00:00
Name: datetime, dtype: str


Missing timestamps:
0

Number of unique timestamps:
21079


In [10]:
# Convert timestamp column to datetime

df["datetime"] = pd.to_datetime(df["datetime"])

# Verify the conversion
print("Updated data type:", df["datetime"].dtype)

display(df["datetime"].head())

Updated data type: datetime64[us]


0   2022-01-01 00:00:00
1   2022-01-01 01:00:00
2   2022-01-01 02:00:00
3   2022-01-01 03:00:00
4   2022-01-01 04:00:00
Name: datetime, dtype: datetime64[us]

In [11]:
# Check for missing timestamps

missing_timestamps = df["datetime"].isna().sum()

print("Missing timestamps:", missing_timestamps)

Missing timestamps: 0


In [12]:
# Check for duplicate timestamps

duplicate_timestamps = df["datetime"].duplicated().sum()

print("Duplicate timestamps:", duplicate_timestamps)

Duplicate timestamps: 0


In [13]:
# Check whether timestamps are chronologically ordered

is_sorted = df["datetime"].is_monotonic_increasing

print("Chronologically sorted:", is_sorted)

Chronologically sorted: True


In [14]:
# Check the time difference between consecutive observations

time_difference = df["datetime"].diff()

missing_hour_gaps = (
    time_difference > pd.Timedelta(hours=1)
).sum()

print("Gaps longer than one hour:", missing_hour_gaps)

Gaps longer than one hour: 14


In [15]:
# Identify gaps between consecutive timestamps

time_difference = df["datetime"].diff()

gap_report = df.loc[
    time_difference > pd.Timedelta(hours=1),
    ["datetime"]
].copy()

gap_report["previous_timestamp"] = (
    df["datetime"].shift(1).loc[gap_report.index]
)

gap_report["gap_duration"] = (
    gap_report["datetime"] - gap_report["previous_timestamp"]
)

gap_report["missing_hours"] = (
    gap_report["gap_duration"] / pd.Timedelta(hours=1) - 1
).astype(int)

# Arrange the report for easy interpretation
gap_report = gap_report[
    [
        "previous_timestamp",
        "datetime",
        "gap_duration",
        "missing_hours"
    ]
]

display(gap_report.reset_index(drop=True))

,previous_timestamp,datetime,gap_duration,missing_hours
0,2022-01-11 06:00:00,2022-01-11 09:00:00,0 days 03:00:00,2
1,2022-01-12 07:00:00,2022-01-12 09:00:00,0 days 02:00:00,1
2,2022-01-29 05:00:00,2022-01-31 10:00:00,2 days 05:00:00,52
3,2022-02-07 07:00:00,2022-02-07 09:00:00,0 days 02:00:00,1
4,2022-02-14 06:00:00,2022-02-14 08:00:00,0 days 02:00:00,1
5,2022-03-03 07:00:00,2022-03-03 09:00:00,0 days 02:00:00,1
6,2022-03-04 07:00:00,2022-03-04 09:00:00,0 days 02:00:00,1
7,2023-02-04 03:00:00,2023-02-04 05:00:00,0 days 02:00:00,1
8,2023-02-28 02:00:00,2023-02-28 04:00:00,0 days 02:00:00,1
9,2023-03-14 03:00:00,2023-03-14 05:00:00,0 days 02:00:00,1


In [16]:
# Calculate total missing hourly observations
expected_timestamps = pd.date_range(
    start=df["datetime"].min(),
    end=df["datetime"].max(),
    freq="h"
)

total_missing_gaps = len(
    expected_timestamps.difference(df["datetime"])
)

print("Total missing hourly observations:", total_missing_gaps)

Total missing hourly observations: 66


In [17]:
# Final validation of timestamp standardisation

assert pd.api.types.is_datetime64_any_dtype(df["datetime"])
assert df["datetime"].notna().all()
assert not df["datetime"].duplicated().any()
assert df["datetime"].is_monotonic_increasing

print("Timestamp standardisation: PASSED")
print("Missing timestamp values:", df["datetime"].isna().sum())
print("Duplicate timestamps:", df["datetime"].duplicated().sum())
print("Chronologically sorted:", df["datetime"].is_monotonic_increasing)
print("Total missing hourly observations:", total_missing_gaps)

Timestamp standardisation: PASSED
Missing timestamp values: 0
Duplicate timestamps: 0
Chronologically sorted: True
Total missing hourly observations: 66


### Datetime Standardisation: Validation and Conclusion

**1. Data Type Standardisation**

* Converted the `datetime` column from string (`str`) to Pandas datetime format (`datetime64[us]`).
* Confirmed that the conversion was successful and preserved the original timestamp values.

**2. Missing Value Validation**

* Confirmed that the column contains **0 missing or invalid timestamp values**.
* No imputation was required for the existing timestamp values.

**3. Duplicate Validation**

* Confirmed that there are **0 duplicate timestamps**.
* Each recorded timestamp uniquely identifies an observation.

**4. Chronological Integrity**

* Verified that observations are arranged in ascending chronological order.
* No additional sorting was required.

**5. Temporal Continuity Assessment**

* Identified **66 missing hourly observations** across the dataset's recorded period.
* Individual gaps will not be investigated or imputed during the current cleaning stage.
* Existing observations will be retained without inserting artificial timestamps or assigning assumed demand values.
* The implications of these gaps will be considered during feature engineering and model development, particularly for lag construction, rolling-window calculations and temporal validation.

**6. Final Decision**

* The `datetime` column meets the current standardisation requirements.
* No further modifications to the timestamp values are required at this stage.
* Temporal gap handling is deferred to the modelling stage, where appropriate safeguards will be applied.



In [18]:
# Extract the hour from the datetime column
df["hour"] = df["datetime"].dt.hour

# Inspect the extracted feature
print("Hour feature created successfully.")
print(f"Data type: {df['hour'].dtype}")
print(f"Minimum hour: {df['hour'].min()}")
print(f"Maximum hour: {df['hour'].max()}")

# Display sample records
df[["datetime", "hour"]].head(10)

Hour feature created successfully.
Data type: int32
Minimum hour: 0
Maximum hour: 23


,datetime,hour
0,2022-01-01 00:00:00,0
1,2022-01-01 01:00:00,1
2,2022-01-01 02:00:00,2
3,2022-01-01 03:00:00,3
4,2022-01-01 04:00:00,4
5,2022-01-01 05:00:00,5
6,2022-01-01 06:00:00,6
7,2022-01-01 07:00:00,7
8,2022-01-01 08:00:00,8
9,2022-01-01 09:00:00,9


In [19]:
df.head(5)

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28.00,97.00,125.00,0
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31.00,61.00,92.00,1
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32.00,41.00,73.00,2
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54.00,41.00,95.00,3
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49.00,42.00,91.00,4


In [20]:
# Extract the day of the month
df["day_of_month"] = df["datetime"].dt.day

# Validate the extracted feature
print("Day-of-month feature created successfully.")
print(f"Data type: {df['day_of_month'].dtype}")
print(f"Minimum day: {df['day_of_month'].min()}")
print(f"Maximum day: {df['day_of_month'].max()}")
print(f"Missing values: {df['day_of_month'].isna().sum()}")

# Inspect sample records
df[["datetime", "day_of_month"]].head(10)

Day-of-month feature created successfully.
Data type: int32
Minimum day: 1
Maximum day: 31
Missing values: 0


,datetime,day_of_month
0,2022-01-01 00:00:00,1
1,2022-01-01 01:00:00,1
2,2022-01-01 02:00:00,1
3,2022-01-01 03:00:00,1
4,2022-01-01 04:00:00,1
5,2022-01-01 05:00:00,1
6,2022-01-01 06:00:00,1
7,2022-01-01 07:00:00,1
8,2022-01-01 08:00:00,1
9,2022-01-01 09:00:00,1


In [21]:
# Extract the weekday number from datetime
df["weekday_num"] = df["datetime"].dt.dayofweek

# Validate the extracted feature
print("Weekday feature created successfully.")
print(f"Data type: {df['weekday_num'].dtype}")
print(f"Minimum weekday: {df['weekday_num'].min()}")
print(f"Maximum weekday: {df['weekday_num'].max()}")
print(f"Missing values: {df['weekday_num'].isna().sum()}")

# Inspect sample records
df[["datetime", "weekday_num"]].head(10)

Weekday feature created successfully.
Data type: int32
Minimum weekday: 0
Maximum weekday: 6
Missing values: 0


,datetime,weekday_num
0,2022-01-01 00:00:00,5
1,2022-01-01 01:00:00,5
2,2022-01-01 02:00:00,5
3,2022-01-01 03:00:00,5
4,2022-01-01 04:00:00,5
5,2022-01-01 05:00:00,5
6,2022-01-01 06:00:00,5
7,2022-01-01 07:00:00,5
8,2022-01-01 08:00:00,5
9,2022-01-01 09:00:00,5


In [22]:
df.head(5)

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour,day_of_month,weekday_num
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28.00,97.00,125.00,0,1,5
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31.00,61.00,92.00,1,1,5
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32.00,41.00,73.00,2,1,5
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54.00,41.00,95.00,3,1,5
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49.00,42.00,91.00,4,1,5


In [23]:
# Inspect the existing and newly extracted weekday features
print("Existing weekday values:")
print(sorted(df["weekday"].dropna().unique()))

print("\nNew weekday_num values:")
print(sorted(df["weekday_num"].dropna().unique()))

print("\nExisting weekday data type:")
print(df["weekday"].dtype)

# Compare the two columns
print("\nExisting weekday value counts:")
print(df["weekday"].value_counts(dropna=False).sort_index())

Existing weekday values:
['FRIDAY', 'MONDAY', 'SATURDAY', 'SUNDAY', 'THURSDAY', 'TUESDAY', 'WEDNESDAY']

New weekday_num values:
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]

Existing weekday data type:
str

Existing weekday value counts:
weekday
FRIDAY       3000
MONDAY       3012
SATURDAY     3005
SUNDAY       2998
THURSDAY     3022
TUESDAY      3020
WEDNESDAY    3022
Name: count, dtype: int64


In [24]:
# ============================================================
# WEEKDAY CONSISTENCY VALIDATION
# ============================================================

# Map weekday numbers to their corresponding names
weekday_mapping = {
    0: "MONDAY",
    1: "TUESDAY",
    2: "WEDNESDAY",
    3: "THURSDAY",
    4: "FRIDAY",
    5: "SATURDAY",
    6: "SUNDAY"
}

# Generate expected weekday names from the datetime column
expected_weekday = df["weekday_num"].map(weekday_mapping)

# Compare existing weekday names with expected weekday names
weekday_mismatches = (
    df["weekday"].str.strip().str.upper() != expected_weekday
)

print("Weekday consistency validation")
print("-" * 40)
print(f"Total records: {len(df)}")
print(f"Mismatched weekday records: {weekday_mismatches.sum()}")
print(f"Matching weekday records: {(~weekday_mismatches).sum()}")
print(f"Missing existing weekday values: {df['weekday'].isna().sum()}")

# Display mismatches if any exist
if weekday_mismatches.any():
    display(
        df.loc[
            weekday_mismatches,
            ["datetime", "weekday", "weekday_num"]
        ].head(10)
    )
else:
    print("All weekday names match their corresponding dates.")


Weekday consistency validation
----------------------------------------
Total records: 21079
Mismatched weekday records: 0
Matching weekday records: 21079
Missing existing weekday values: 0
All weekday names match their corresponding dates.


In [25]:
# Extract the month from datetime
df["month"] = df["datetime"].dt.month

# Validate the extracted feature
print("Month feature created successfully.")
print(f"Data type: {df['month'].dtype}")
print(f"Minimum month: {df['month'].min()}")
print(f"Maximum month: {df['month'].max()}")
print(f"Missing values: {df['month'].isna().sum()}")

# Inspect sample records
df[["datetime", "month"]].head(10)

Month feature created successfully.
Data type: int32
Minimum month: 1
Maximum month: 12
Missing values: 0


,datetime,month
0,2022-01-01 00:00:00,1
1,2022-01-01 01:00:00,1
2,2022-01-01 02:00:00,1
3,2022-01-01 03:00:00,1
4,2022-01-01 04:00:00,1
5,2022-01-01 05:00:00,1
6,2022-01-01 06:00:00,1
7,2022-01-01 07:00:00,1
8,2022-01-01 08:00:00,1
9,2022-01-01 09:00:00,1


In [26]:

# Extract the year from datetime
df["year"] = df["datetime"].dt.year

# Validate the extracted feature
print("Year feature created successfully.")
print(f"Data type: {df['year'].dtype}")
print(f"Minimum year: {df['year'].min()}")
print(f"Maximum year: {df['year'].max()}")
print(f"Missing values: {df['year'].isna().sum()}")

# Inspect sample records
df[["datetime", "year"]].head(10)



Year feature created successfully.
Data type: int32
Minimum year: 2022
Maximum year: 2024
Missing values: 0


,datetime,year
0,2022-01-01 00:00:00,2022
1,2022-01-01 01:00:00,2022
2,2022-01-01 02:00:00,2022
3,2022-01-01 03:00:00,2022
4,2022-01-01 04:00:00,2022
5,2022-01-01 05:00:00,2022
6,2022-01-01 06:00:00,2022
7,2022-01-01 07:00:00,2022
8,2022-01-01 08:00:00,2022
9,2022-01-01 09:00:00,2022


In [27]:
df.head()

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour,day_of_month,weekday_num,month,year
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28.00,97.00,125.00,0,1,5,1,2022
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31.00,61.00,92.00,1,1,5,1,2022
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32.00,41.00,73.00,2,1,5,1,2022
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54.00,41.00,95.00,3,1,5,1,2022
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49.00,42.00,91.00,4,1,5,1,2022


In [28]:
# Review the extracted datetime features
datetime_features = [
    "datetime",
    "hour",
    "day_of_month",
    "weekday",
    "weekday_num",
    "month",
    "year",
    "weekend",
    "holiday"
]

# Display available datetime features
display(df[datetime_features].head(10))

# Check missing values in these features
print("\nMissing values in datetime features:")
display(df[datetime_features].isna().sum())

# Check the dataset dimensions
print(f"\nDataset shape: {df.shape}")



,datetime,hour,day_of_month,weekday,weekday_num,month,year,weekend,holiday
0,2022-01-01 00:00:00,0,1,SATURDAY,5,1,2022,1,1
1,2022-01-01 01:00:00,1,1,SATURDAY,5,1,2022,1,1
2,2022-01-01 02:00:00,2,1,SATURDAY,5,1,2022,1,1
3,2022-01-01 03:00:00,3,1,SATURDAY,5,1,2022,1,1
4,2022-01-01 04:00:00,4,1,SATURDAY,5,1,2022,1,1
5,2022-01-01 05:00:00,5,1,SATURDAY,5,1,2022,1,1
6,2022-01-01 06:00:00,6,1,SATURDAY,5,1,2022,1,1
7,2022-01-01 07:00:00,7,1,SATURDAY,5,1,2022,1,1
8,2022-01-01 08:00:00,8,1,SATURDAY,5,1,2022,1,1
9,2022-01-01 09:00:00,9,1,SATURDAY,5,1,2022,1,1



Missing values in datetime features:


datetime        0
hour            0
day_of_month    0
weekday         0
weekday_num     0
month           0
year            0
weekend         0
holiday         0
dtype: int64


Dataset shape: (21079, 20)


In [29]:
# ============================================================
# VALIDATE WEEKEND INDICATOR
# ============================================================

# Derive the expected weekend indicator from datetime
expected_weekend = (
    df["datetime"].dt.dayofweek >= 5
).astype("int8")

# Compare the existing indicator with the expected values
weekend_mismatches = df["weekend"] != expected_weekend

print("Weekend indicator validation")
print("-" * 40)
print(f"Total records: {len(df)}")
print(f"Mismatched records: {weekend_mismatches.sum()}")
print(f"Matching records: {(~weekend_mismatches).sum()}")
print(f"Missing weekend values: {df['weekend'].isna().sum()}")

# Show examples if inconsistencies exist
if weekend_mismatches.any():
    display(
        df.loc[
            weekend_mismatches,
            ["datetime", "day_of_week", "weekend"]
        ].head(10)
    )
else:
    print("All weekend indicators match their timestamps.")



Weekend indicator validation
----------------------------------------
Total records: 21079
Mismatched records: 0
Matching records: 21079
Missing weekend values: 0
All weekend indicators match their timestamps.


In [30]:
# ============================================================
# VALIDATE HOLIDAY INDICATOR
# ============================================================

# Inspect the holiday indicator
print("Holiday indicator data type:")
print(df["holiday"].dtype)

print("\nUnique holiday values:")
print(sorted(df["holiday"].dropna().unique()))

print("\nHoliday value counts:")
print(df["holiday"].value_counts(dropna=False).sort_index())

print("\nHoliday proportions (%):")
print(
    (df["holiday"].value_counts(normalize=True, dropna=False)
     .sort_index() * 100).round(2)
)


Holiday indicator data type:
int64

Unique holiday values:
[np.int64(0), np.int64(1)]

Holiday value counts:
holiday
0    20191
1      888
Name: count, dtype: int64

Holiday proportions (%):
holiday
0   95.79
1    4.21
Name: proportion, dtype: float64


In [31]:
# Inspect all columns and their data types
print("Dataset shape:", df.shape)

print("\nColumn names and data types:")
display(
    df.dtypes.rename("data_type")
      .to_frame()
      .reset_index(names="column_name")
)

print("\nNumber of columns by data type:")
print(df.dtypes.astype(str).value_counts())


Dataset shape: (21079, 20)

Column names and data types:


,column_name,data_type
0,datetime,datetime64[us]
1,weekday,str
2,weekend,int64
3,holiday,int64
4,casual_duration,float64
5,member_duration,float64
6,total_duration,float64
7,temperature,float64
8,humidity,float64
9,windspeed,float64



Number of columns by data type:
float64           10
int32              5
str                2
int64              2
datetime64[us]     1
Name: count, dtype: int64


In [32]:
# ============================================================
# VALIDATE RIDER-COUNT COLUMNS
# ============================================================

# Identify rider-count columns
count_columns = [
    "casual_riders",
    "member_riders",
    "total_riders"
]

# Build a data-quality report
count_quality_report = pd.DataFrame({
    "data_type": df[count_columns].dtypes.astype(str),
    "missing_values": df[count_columns].isna().sum(),
    "negative_values": (df[count_columns] < 0).sum(),
    "zero_values": (df[count_columns] == 0).sum(),
    "fractional_values": (
        (df[count_columns].notna()) &
        (df[count_columns] % 1 != 0)
    ).sum(),
    "unique_values": df[count_columns].nunique(),
    "minimum": df[count_columns].min(),
    "maximum": df[count_columns].max()
})

display(count_quality_report)


,data_type,missing_values,negative_values,zero_values,fractional_values,unique_values,minimum,maximum
casual_riders,float64,0,0,325,0,681,0.00,914.00
member_riders,float64,0,0,11,0,1486,0.00,2434.00
total_riders,float64,0,0,0,0,1794,1.00,2744.00


In [33]:
# Identify fractional observations in rider-count columns
count_columns = [
    "casual_riders",
    "member_riders",
    "total_riders"
]

for col in count_columns:
    fractional_mask = (
        df[col].notna() &
        (df[col] % 1 != 0)
    )

    print(f"\n{'=' * 55}")
    print(f"Column: {col}")
    print(f"Fractional observations: {fractional_mask.sum()}")

    if fractional_mask.any():
        display(
            df.loc[
                fractional_mask,
                ["datetime", col, "casual_riders",
                 "member_riders", "total_riders"]
            ].head(10)
        )
    else:
        print("No fractional values found.")



Column: casual_riders
Fractional observations: 0
No fractional values found.

Column: member_riders
Fractional observations: 0
No fractional values found.

Column: total_riders
Fractional observations: 0
No fractional values found.


In [34]:
# Identify rider-count columns
count_columns = [
    "casual_riders",
    "member_riders",
    "total_riders"
]

# Convert whole-number rider counts to int64
df[count_columns] = df[count_columns].astype("int64")

# Validate the conversion
print("Rider-count data type conversion completed.")
print("\nUpdated data types:")
print(df[count_columns].dtypes)

print("\nValidation:")
print("Missing values:")
print(df[count_columns].isna().sum())

print("\nFractional values:")
print((df[count_columns] % 1 != 0).sum())

print("\nSample records:")
display(df[count_columns].head(10))


Rider-count data type conversion completed.

Updated data types:
casual_riders    int64
member_riders    int64
total_riders     int64
dtype: object

Validation:
Missing values:
casual_riders    0
member_riders    0
total_riders     0
dtype: int64

Fractional values:
casual_riders    0
member_riders    0
total_riders     0
dtype: int64

Sample records:


,casual_riders,member_riders,total_riders
0,28,97,125
1,31,61,92
2,32,41,73
3,54,41,95
4,49,42,91
5,62,62,124
6,121,53,174
7,68,41,109
8,29,14,43
9,9,4,13


In [35]:
# Calculate total riders from the component counts
calculated_total = (
    df["casual_riders"] + df["member_riders"]
)

# Compare the calculated total with the recorded total
total_mismatches = df["total_riders"] != calculated_total

print("Total rider-count reconciliation")
print("-" * 45)
print(f"Total records: {len(df)}")
print(f"Matching records: {(~total_mismatches).sum()}")
print(f"Mismatched records: {total_mismatches.sum()}")
print(
    "Match rate: "
    f"{(~total_mismatches).mean() * 100:.2f}%"
)

# Display mismatches, if any
if total_mismatches.any():
    display(
        df.loc[
            total_mismatches,
            [
                "datetime",
                "casual_riders",
                "member_riders",
                "total_riders"
            ]
        ].head(10)
    )
else:
    print("All total rider counts reconcile correctly.")


Total rider-count reconciliation
---------------------------------------------
Total records: 21079
Matching records: 21079
Mismatched records: 0
Match rate: 100.00%
All total rider counts reconcile correctly.


In [36]:
# ============================================================
# VALIDATE RIDER DURATION COLUMNS
# ============================================================

# Identify duration columns
duration_columns = [
    "casual_duration",
    "member_duration",
    "total_duration"
]

# Generate a duration data-quality report
duration_report = pd.DataFrame({
    "data_type": df[duration_columns].dtypes.astype(str),
    "missing_values": df[duration_columns].isna().sum(),
    "negative_values": (df[duration_columns] < 0).sum(),
    "zero_values": (df[duration_columns] == 0).sum(),
    "unique_values": df[duration_columns].nunique(),
    "minimum": df[duration_columns].min(),
    "median": df[duration_columns].median(),
    "mean": df[duration_columns].mean(),
    "maximum": df[duration_columns].max()
})

display(duration_report)


,data_type,missing_values,negative_values,zero_values,unique_values,minimum,median,mean,maximum
casual_duration,float64,0,0,325,19921,0.00,102418.00,411039.12,18528492.00
member_duration,float64,0,0,11,20533,0.00,188858.00,311164.38,6981540.00
total_duration,float64,0,0,0,20810,156.00,339895.00,722203.50,19330450.00


In [37]:

# Generate a clearly formatted duration summary
duration_columns = [
    "casual_duration",
    "member_duration",
    "total_duration"
]

for col in duration_columns:
    print(f"\n{'=' * 55}")
    print(f"Column: {col}")
    print(f"Data type: {df[col].dtype}")
    print(f"Missing values: {df[col].isna().sum()}")
    print(f"Negative values: {(df[col] < 0).sum()}")
    print(f"Zero values: {(df[col] == 0).sum()}")
    print(f"Unique values: {df[col].nunique()}")
    print("\nDescriptive statistics:")
    display(df[col].describe().to_frame(name="value"))




Column: casual_duration
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 325
Unique values: 19921

Descriptive statistics:


,value
count,21079.00
mean,411039.12
std,834119.35
min,0.00
25%,20646.00
50%,102418.00
75%,393061.50
max,18528492.00



Column: member_duration
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 11
Unique values: 20533

Descriptive statistics:


,value
count,21079.00
mean,311164.38
std,432014.83
min,0.00
25%,47849.50
50%,188858.00
75%,404047.50
max,6981540.00



Column: total_duration
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 0
Unique values: 20810

Descriptive statistics:


,value
count,21079.00
mean,722203.50
std,1062145.82
min,156.00
25%,86993.50
50%,339895.00
75%,886450.50
max,19330450.00


In [38]:
# Descriptive statistics for member duration
print("MEMBER DURATION QUALITY REPORT")
print("=" * 45)

print(f"Observations: {df['member_duration'].count():,}")
print(f"Missing values: {df['member_duration'].isna().sum():,}")
print(f"Negative values: {(df['member_duration'] < 0).sum():,}")
print(f"Zero values: {(df['member_duration'] == 0).sum():,}")
print(f"Unique values: {df['member_duration'].nunique():,}")

display(df["member_duration"].describe().to_frame(name="value"))



MEMBER DURATION QUALITY REPORT
Observations: 21,079
Missing values: 0
Negative values: 0
Zero values: 11
Unique values: 20,533


,value
count,21079.00
mean,311164.38
std,432014.83
min,0.00
25%,47849.50
50%,188858.00
75%,404047.50
max,6981540.00


In [39]:
# Descriptive statistics for total duration
print("TOTAL DURATION QUALITY REPORT")
print("=" * 45)

print(f"Observations: {df['total_duration'].count():,}")
print(f"Missing values: {df['total_duration'].isna().sum():,}")
print(f"Negative values: {(df['total_duration'] < 0).sum():,}")
print(f"Zero values: {(df['total_duration'] == 0).sum():,}")
print(f"Unique values: {df['total_duration'].nunique():,}")

display(df["total_duration"].describe().to_frame(name="value"))


TOTAL DURATION QUALITY REPORT
Observations: 21,079
Missing values: 0
Negative values: 0
Zero values: 0
Unique values: 20,810


,value
count,21079.00
mean,722203.50
std,1062145.82
min,156.00
25%,86993.50
50%,339895.00
75%,886450.50
max,19330450.00


In [40]:
# Calculate total duration from the component durations
calculated_duration = (
    df["casual_duration"] + df["member_duration"]
)

# Calculate the difference between recorded and calculated totals
duration_difference = (
    df["total_duration"] - calculated_duration
)

# Identify exact mismatches
duration_mismatches = ~(
    df["total_duration"].eq(calculated_duration)
)

print("DURATION RECONCILIATION REPORT")
print("=" * 45)
print(f"Total records: {len(df):,}")
print(f"Matching records: {(~duration_mismatches).sum():,}")
print(f"Mismatched records: {duration_mismatches.sum():,}")
print(
    f"Match rate: {(~duration_mismatches).mean() * 100:.2f}%"
)

print("\nDifference statistics:")
display(duration_difference.describe().to_frame(name="difference"))

# Inspect a sample of mismatches, if any
if duration_mismatches.any():
    print("\nSample mismatched records:")
    display(
        df.loc[
            duration_mismatches,
            [
                "datetime",
                "casual_duration",
                "member_duration",
                "total_duration"
            ]
        ].head(10)
    )
else:
    print("\nAll duration totals reconcile exactly.")



DURATION RECONCILIATION REPORT
Total records: 21,079
Matching records: 21,079
Mismatched records: 0
Match rate: 100.00%

Difference statistics:


,difference
count,21079.00
mean,0.00
std,0.00
min,0.00
25%,0.00
50%,0.00
75%,0.00
max,0.00



All duration totals reconcile exactly.


In [41]:
df.head(5)

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour,day_of_month,weekday_num,month,year
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28,97,125,0,1,5,1,2022
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31,61,92,1,1,5,1,2022
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32,41,73,2,1,5,1,2022
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54,41,95,3,1,5,1,2022
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49,42,91,4,1,5,1,2022


In [42]:
# ============================================================
# 4.2 WEATHER CONDITION INSPECTION
# ============================================================

# Inspect the weather condition feature
print("WEATHER CONDITION INSPECTION")
print("=" * 45)

print(f"Data type: {df['weather_condition'].dtype}")
print(f"Total observations: {len(df):,}")
print(f"Missing values: {df['weather_condition'].isna().sum():,}")
print(f"Unique categories: {df['weather_condition'].nunique():,}")

print("\nUnique weather categories:")
print(sorted(df["weather_condition"].dropna().unique()))

print("\nCategory frequencies:")
display(
    df["weather_condition"]
      .value_counts(dropna=False)
      .rename_axis("weather_condition")
      .reset_index(name="frequency")
)


WEATHER CONDITION INSPECTION
Data type: str
Total observations: 21,079
Missing values: 0
Unique categories: 5

Unique weather categories:
['Clear', 'Cloudy', 'Fair', 'Rain', 'Snow']

Category frequencies:


,weather_condition,frequency
0,Cloudy,10164
1,Fair,6800
2,Rain,2414
3,Clear,1282
4,Snow,419


In [43]:
# Standardise whitespace and capitalisation
df["weather_condition"] = (
    df["weather_condition"]
    .str.strip()
    .str.title()
)

# Validate the standardised categories
print("Weather label standardisation completed.")
print(f"Missing values: {df['weather_condition'].isna().sum()}")
print(f"Unique categories: {df['weather_condition'].nunique()}")

print("\nStandardised categories:")
print(sorted(df["weather_condition"].dropna().unique()))

print("\nUpdated category frequencies:")
display(
    df["weather_condition"]
      .value_counts()
      .rename_axis("weather_condition")
      .reset_index(name="frequency")
)


Weather label standardisation completed.
Missing values: 0
Unique categories: 5

Standardised categories:
['Clear', 'Cloudy', 'Fair', 'Rain', 'Snow']

Updated category frequencies:


,weather_condition,frequency
0,Cloudy,10164
1,Fair,6800
2,Rain,2414
3,Clear,1282
4,Snow,419


In [44]:
# ============================================================
# INSPECTING THE ENVIRONMENTAL FEATURES
# ============================================================

# Identify environmental features
environmental_columns = [
    "temperature",
    "humidity",
    "windspeed",
    "pressure"
]

# Generate an initial data-quality report
environmental_report = pd.DataFrame({
    "data_type": df[environmental_columns].dtypes.astype(str),
    "missing_values": df[environmental_columns].isna().sum(),
    "negative_values": (df[environmental_columns] < 0).sum(),
    "zero_values": (df[environmental_columns] == 0).sum(),
    "unique_values": df[environmental_columns].nunique(),
    "minimum": df[environmental_columns].min(),
    "median": df[environmental_columns].median(),
    "mean": df[environmental_columns].mean(),
    "maximum": df[environmental_columns].max()
})

display(environmental_report)


,data_type,missing_values,negative_values,zero_values,unique_values,minimum,median,mean,maximum
temperature,float64,0,2115,284,127,-22.80,10.60,11.15,37.20
humidity,float64,0,0,0,85,13.00,68.00,67.30,100.00
windspeed,float64,0,0,609,54,0.00,16.60,17.26,79.60
pressure,float64,0,0,0,545,980.20,1015.80,1015.75,1042.30


In [45]:

# Recalculate environmental feature statistics individually
environmental_columns = [
    "temperature",
    "humidity",
    "windspeed",
    "pressure"
]

for col in environmental_columns:
    print(f"\n{'=' * 50}")
    print(f"Feature: {col}")
    print(f"Data type: {df[col].dtype}")
    print(f"Missing values: {df[col].isna().sum():,}")
    print(f"Negative values: {(df[col] < 0).sum():,}")
    print(f"Zero values: {(df[col] == 0).sum():,}")
    print(f"Unique values: {df[col].nunique():,}")

    print("\nDescriptive statistics:")
    display(df[col].describe().to_frame(name="value"))



Feature: temperature
Data type: float64
Missing values: 0
Negative values: 2,115
Zero values: 284
Unique values: 127

Descriptive statistics:


,value
count,21079.00
mean,11.15
std,9.20
min,-22.80
25%,4.40
50%,10.60
75%,18.30
max,37.20



Feature: humidity
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 0
Unique values: 85

Descriptive statistics:


,value
count,21079.00
mean,67.30
std,20.17
min,13.00
25%,52.00
50%,68.00
75%,84.00
max,100.00



Feature: windspeed
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 609
Unique values: 54

Descriptive statistics:


,value
count,21079.00
mean,17.26
std,8.32
min,0.00
25%,11.20
50%,16.60
75%,22.30
max,79.60



Feature: pressure
Data type: float64
Missing values: 0
Negative values: 0
Zero values: 0
Unique values: 545

Descriptive statistics:


,value
count,21079.00
mean,1015.75
std,8.56
min,980.20
25%,1010.50
50%,1015.80
75%,1021.40
max,1042.30




| Feature       | Missing values | Negative values | Zero values | Minimum |   Median |  Maximum | Initial assessment                                               |
| ------------- | -------------: | --------------: | ----------: | ------: | -------: | -------: | ---------------------------------------------------------------- |
| `temperature` |              0 |           2,115 |         284 |  -22.80 |    10.60 |    37.20 | Negative and zero values are valid temperatures.                 |
| `humidity`    |              0 |               0 |           0 |   13.00 |    68.00 |   100.00 | Values are within the expected percentage range.                 |
| `windspeed`   |              0 |               0 |         609 |    0.00 |    16.60 |    79.60 | Zero windspeed is valid; maximum requires an outlier assessment. |
| `pressure`    |              0 |               0 |           0 |  980.20 | 1,015.80 | 1,042.30 | Values appear plausible for atmospheric pressure in hPa.         |

### What should we do with each feature?

**1. `temperature`**

* Keep negative values; sub-zero temperatures are physically possible.
* Keep zero values; 0°C is a valid temperature.
* No missing-value treatment is required.
* Do not remove extreme temperatures solely because they are unusual.

**2. `humidity`**

* All values fall between 13 and 100.
* No missing values or zero values exist.
* Retain the feature without modification.

**3. `windspeed`**

* Zero values can represent calm conditions.
* The maximum of 79.60 is considerably higher than the median of 16.60.
* We should investigate whether the maximum represents a legitimate extreme or a potential data-quality issue before deciding whether any action is necessary.

**4. `pressure`**

* The values range from 980.20 to 1,042.30, with a median of 1,015.80.
* These values appear plausible for atmospheric pressure measured in hPa.
* No missing-value treatment or immediate transformation is necessary.



In [46]:
df.head(5)

,datetime,weekday,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour,day_of_month,weekday_num,month,year
0,2022-01-01 00:00:00,SATURDAY,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28,97,125,0,1,5,1,2022
1,2022-01-01 01:00:00,SATURDAY,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31,61,92,1,1,5,1,2022
2,2022-01-01 02:00:00,SATURDAY,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32,41,73,2,1,5,1,2022
3,2022-01-01 03:00:00,SATURDAY,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54,41,95,3,1,5,1,2022
4,2022-01-01 04:00:00,SATURDAY,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49,42,91,4,1,5,1,2022


In [47]:
# Remove the redundant weekday column

df.drop(columns=["weekday"], inplace=True)

# Confirm the removal
print("'weekday' column removed successfully.")
print(f"Dataset shape: {df.shape}")

print("\nRemaining weekday-related columns:")
print([col for col in df.columns if "weekday" in col.lower()])

# Display the remaining columns
display(df.head())

'weekday' column removed successfully.
Dataset shape: (21079, 19)

Remaining weekday-related columns:
['weekday_num']


,datetime,weekend,holiday,casual_duration,member_duration,total_duration,temperature,humidity,windspeed,pressure,weather_condition,casual_riders,member_riders,total_riders,hour,day_of_month,weekday_num,month,year
0,2022-01-01 00:00:00,1,1,554672.00,96509.00,651181.00,6.70,89.00,7.60,1014.90,Cloudy,28,97,125,0,1,5,1,2022
1,2022-01-01 01:00:00,1,1,46276.00,58472.00,104748.00,6.70,89.00,7.60,1014.70,Cloudy,31,61,92,1,1,5,1,2022
2,2022-01-01 02:00:00,1,1,377839.00,45191.00,423030.00,6.70,89.00,7.60,1014.70,Cloudy,32,41,73,2,1,5,1,2022
3,2022-01-01 03:00:00,1,1,2537734.00,37167.00,2574901.00,6.70,96.00,5.40,1014.20,Cloudy,54,41,95,3,1,5,1,2022
4,2022-01-01 04:00:00,1,1,911448.00,43178.00,954626.00,7.20,93.00,5.40,1014.40,Cloudy,49,42,91,4,1,5,1,2022


In [48]:
# 1. Rename columns using consistent, descriptive names

column_mapping = {
    "day_of_month": "day",
    "weekday_num": "weekday",
    "weekend": "weekend",
    "holiday": "holiday",
    "temperature": "temp_c",
    "windspeed": "wind_kmh",
    "pressure": "pressure_hpa",
    "weather_condition": "weather"
}

df.rename(columns=column_mapping, inplace=True)


# 2. Arrange columns into logical groups

column_order = [
    # Time features
    "datetime",
    "hour",
    "day",
    "weekday",
    "weekend",
    "holiday",
    "month",
    "year",

    # Weather features
    "temp_c",
    "humidity",
    "wind_kmh",
    "pressure_hpa",
    "weather",

    # Rider duration
    "casual_duration",
    "member_duration",
    "total_duration",

    # Rider composition
    "casual_riders",
    "member_riders",

    # Target variable — always last
    "total_riders"
]

df = df[column_order]


# 3. Verify the result

print("Dataset shape:", df.shape)
print("\nColumn order:")
for position, column in enumerate(df.columns, start=1):
    print(f"{position:02d}. {column}")

print("\nTarget variable:", df.columns[-1])
display(df.head())

Dataset shape: (21079, 19)

Column order:
01. datetime
02. hour
03. day
04. weekday
05. weekend
06. holiday
07. month
08. year
09. temp_c
10. humidity
11. wind_kmh
12. pressure_hpa
13. weather
14. casual_duration
15. member_duration
16. total_duration
17. casual_riders
18. member_riders
19. total_riders

Target variable: total_riders


,datetime,hour,day,weekday,weekend,holiday,month,year,temp_c,humidity,wind_kmh,pressure_hpa,weather,casual_duration,member_duration,total_duration,casual_riders,member_riders,total_riders
0,2022-01-01 00:00:00,0,1,5,1,1,1,2022,6.70,89.00,7.60,1014.90,Cloudy,554672.00,96509.00,651181.00,28,97,125
1,2022-01-01 01:00:00,1,1,5,1,1,1,2022,6.70,89.00,7.60,1014.70,Cloudy,46276.00,58472.00,104748.00,31,61,92
2,2022-01-01 02:00:00,2,1,5,1,1,1,2022,6.70,89.00,7.60,1014.70,Cloudy,377839.00,45191.00,423030.00,32,41,73
3,2022-01-01 03:00:00,3,1,5,1,1,1,2022,6.70,96.00,5.40,1014.20,Cloudy,2537734.00,37167.00,2574901.00,54,41,95
4,2022-01-01 04:00:00,4,1,5,1,1,1,2022,7.20,93.00,5.40,1014.40,Cloudy,911448.00,43178.00,954626.00,49,42,91


In [49]:
print("DataFrame available:", "df" in globals())

DataFrame available: True


In [52]:
# Define the output path relative to the notebook directory
output_path = Path("data/02_intermediate/blue_bike_boston_clean.parquet")

# Save the cleaned dataset
df.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print("Dataset saved successfully!")
print(f"File: {output_path.resolve()}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Dataset saved successfully!
File: C:\Users\George\boston-bike-demand-forecasting\data\02_intermediate\blue_bike_boston_clean.parquet
Rows: 21,079
Columns: 19


In [53]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal

# Define the saved dataset path
output_path = Path("data/02_intermediate/blue_bike_boston_clean.parquet")

# Load the saved dataset
df_check = pd.read_parquet(output_path, engine="pyarrow")

# Verify the saved data matches the original DataFrame
assert_frame_equal(df, df_check, check_dtype=True)

print("PARQUET VALIDATION: PASSED")
print(f"File exists: {output_path.exists()}")
print(f"File size: {output_path.stat().st_size / (1024 ** 2):.2f} MB")
print(f"Rows: {df_check.shape[0]:,}")
print(f"Columns: {df_check.shape[1]}")
print("All values, column names, and data types match.")

PARQUET VALIDATION: PASSED
File exists: True
File size: 0.78 MB
Rows: 21,079
Columns: 19
All values, column names, and data types match.
